# Health Trainer — Colab Training Runner

Thin runner: the real logic lives in `ml/src/` (source of truth). This notebook only
mounts Drive, unzips the code, and calls the CLI scripts against the
`health_training/` workspace. See `docs/colab-drive-workflow.md`.

**Drive workspace** `health_training/` already has: `colab/ code/ data/{raw,landmarks,splits} runs/ exports/latest`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
DATA_DIR   = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
EXPORT_DIR = f'{DRIVE_ROOT}/exports/latest'
print(DRIVE_ROOT)

In [ ]:
# Unzip the local-authored code (source of truth) uploaded to Drive code/.
!mkdir -p /content/health_trainer_ml
!unzip -o "$DRIVE_ROOT/code/health_trainer_ml.zip" -d /content/health_trainer_ml
%cd /content/health_trainer_ml
!pip install -q -r ml/requirements-colab.txt

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/health_trainer_ml/ml/src'

# 1) extract landmarks  2) build features  3) train  4) evaluate  5) export
!python ml/src/extract_landmarks.py --input-dir "$DATA_DIR/raw/public" \
    --output-dir "$DATA_DIR/landmarks/public" --model pose_landmarker_lite.task
!python ml/src/build_features.py --landmarks-dir "$DATA_DIR/landmarks/public" \
    --config ml/configs/exercise_classifier.yaml --out "$DATA_DIR/splits/exercise_train.npz"
!python ml/src/train_exercise_classifier.py --features "$DATA_DIR/splits/exercise_train.npz" \
    --run-dir "$RUNS_DIR/exercise_classifier_rf_v1" --model baseline
!python ml/src/evaluate.py --features "$DATA_DIR/splits/exercise_train.npz" \
    --run-dir "$RUNS_DIR/exercise_classifier_rf_v1" --n-classes 4

In [ ]:
# Sequence model + TFLite export (GPU). Only export to exports/latest if it clears
# the integration thresholds in docs/colab-drive-workflow.md.
!python ml/src/train_exercise_classifier.py --features "$DATA_DIR/splits/exercise_train.npz" \
    --run-dir "$RUNS_DIR/exercise_classifier_lstm_v1" --model sequence
!python ml/src/export_tflite.py --run-dir "$RUNS_DIR/exercise_classifier_lstm_v1" \
    --export-dir "$EXPORT_DIR"